In [ ]:
from __future__ import annotations
import datetime as dt
from random import random
import sys

import apsw

from tuplesaver.engine import Engine
from tuplesaver.model import Row, TableRow

# Models

In [ ]:
class MyModel(TableRow):
    name: str
    date: dt.datetime
    score: float

Connect to the database and create tables with an `Engine`

In [ ]:
engine = Engine(":memory:")
# engine = Engine("example.db")
engine.ensure_table_created(MyModel)
engine.connection # just the real apsw connection object


In [ ]:
def trace_sql_in_color(cursor, sql, params):
    print(f"\033[94m{sql}\033[0m")
    if params:
        # Get the !r (repr) of each param value individually
        if isinstance(params, dict):
            print(*(f"\033[1;96m{k}: {v!r}\033[0m" for k, v in params.items()), sep=", ")
        else:
            print(*(f"\033[1;96m{p!r}\033[0m" for p in params), sep=", ")
        print()  # Newline after printing all params
    return True

engine.connection.exec_trace = trace_sql_in_color

# Basic CRUD

## Insert row

In [ ]:
row =engine.insert(MyModel("Bart", dt.datetime.now(), 6.5))
row

## Find row by id

In [ ]:
engine.find(MyModel, row.id)

## Find row by Expr

In [ ]:
engine.find(MyModel, MyModel.name == "Bart")

## Select rows

In [ ]:
engine.select(MyModel, MyModel.score > 99.95)

## Update rows

In [ ]:
engine.update(MyModel, t"{MyModel.name} LIKE 'B%'" and MyModel.score < 1, score=99.9)

## Delete rows

In [ ]:
engine.delete(MyModel, row.id)

# Foreign Keys Relationships
Models can be related by using a model as a field type in another model.

In [ ]:
class Band(TableRow):
    name: str
    active: bool

class BandMember(TableRow):
    band: Band
    name: str
    instrument: str

engine.ensure_table_created(Band)
engine.ensure_table_created(BandMember)

You can save a model and then use it as a related field in another model.

In [ ]:
devo = engine.insert(Band("Devo", True))
mark = engine.insert(BandMember(devo, "Mark Mothersbaugh", "Keyboards"))

## Lazy loading
FK related models are lazy loaded by default.

In [ ]:
singer = engine.find(BandMember, mark.id)

print(singer.name)
print(singer.band.name) # `band` field gets lazy loaded

Dummy data for later, also demonstrates multi-level relationships.

In [ ]:
class League(TableRow):
    leaguename: str

class Team(TableRow):
    teamname: str
    league: League

class Athlete(TableRow):
    name: str
    team: Team


engine.ensure_table_created(League)
engine.ensure_table_created(Team)
engine.ensure_table_created(Athlete)

with engine.connection:  # transaction
    # Insert dummy data
    leagues = [
        engine.insert(League("Big")),
        engine.insert(League("Small")),
        ]
    teams = [
        engine.insert(Team("Red", leagues[0])),
        engine.insert(Team("Ramble", leagues[1])),
        engine.insert(Team("Blue", leagues[0])),
        engine.insert(Team("Green", leagues[1])),
        ]
    players = [
        alice:=engine.insert(Athlete("Alice", teams[0])),
        engine.insert(Athlete("Bob", teams[0])),
        engine.insert(Athlete("Charlie", teams[1])),
        engine.insert(Athlete("Dave", teams[2])),
        engine.insert(Athlete("Beth", teams[3])),
        engine.insert(Athlete("Frank", teams[2])),
    ]

In [ ]:
athlete = engine.find(Athlete, Athlete.name == "Alice")
athlete.team.league.leaguename # multi-step lazy loading

# Querying

The raw interface for querying is very simple, you can use any arbitrary SQL directly.

```python
engine.query(Model, sql, params)
```

In [ ]:
# insert some data since we deleted all MyModels earlier
engine.insert(MyModel("Milhouse", dt.datetime.now(), 2.0))
engine.insert(MyModel("Maggie", dt.datetime.now(), 10.0))


class AverageScoreResults(Row):
    avg_score: float
    scorecount: int

sql = 'SELECT avg(score), count(*) FROM MyModel'

result = engine.query(AverageScoreResults, sql).fetchone()
assert result is not None

print(f'The table has {result.scorecount} rows, with an average of {result.avg_score:0.2f}')

`engine.select` is a simple yet powerful way to retrieve Models from the database.

The most simple case selects all rows from a table.


In [ ]:
for player in engine.select(Athlete):
    print(player.name)

You can use Model relation expressions to add `WHERE` clauses.

In [ ]:
for player in engine.select(Athlete, Athlete.name == 'Beth'):
    print(player.name)

Queries spanning across relations are automatically generated using `EXISTS` semi-joins safely to prevent fanout.

In [ ]:
for player in engine.select(Athlete, Athlete.team.teamname == 'Red'):
    print(player.name)

for player in engine.select(Athlete, Athlete.team.league.leaguename == 'Big'):
    print(player.name)

You can also pass t-string based expressions for more complex predicates.

In [ ]:
for player in engine.select(Athlete, t'{Athlete.team.league.leaguename} LIKE "B%"'):
    print(player.name)

# Querys requiring backrefs

Eventually we will support backrefs:

```python
class League(Row):
    id: int | None
    leaguename: str
    teams: BackRef[Team] # backref will be something like this
```

And then you could do:

```python
engine.select(League, League.teams.teamname == "Big")
```

For now just fallback to raw queries:


In [ ]:
sql = """
SELECT * FROM League
JOIN Team ON Team.league = League.id
WHERE Team.teamname = 'Big'
"""

engine.query(League, sql).fetchone()

## Ad-hoc models and the `Any` type
Models can also be completely ad-hoc, instead of `TableRow`, they are merely `Row`. These models do not have an `id` field. They also have a special field type available, `Any`, which can be used to represent any type of data. This is particularly useful for dynamic or polymorphic data structures where the exact type may not be known until runtime.


In [ ]:
from typing import Any

class TableInfo(Row):
    cid: int
    name: str
    type: str
    notnull: int
    dflt_value: Any # `Any` will return raw value matching python's bare sqlite3, without conversion
    pk: int

sql = f"PRAGMA table_info({Athlete.__name__})"

cols = engine.query(TableInfo, sql).fetchall()
for col in cols:
    print(f"{col.cid:2d} {col.name:10s} {col.type:10s} {str(col.dflt_value or 'None'):10s}")

## SQLite3 Cursor
`Engine.query`returns a real `apsw.Cursor`, you can use it to `fetchall`, `fetchone`, `fetchmany`, etc.

In [ ]:
row = engine.query(Athlete).fetchone()

# Persisting Native and Advanced Python Types
TupleSaver supports a wide variety of standard python types and automatically maps them to SQLite storage. For example, `datetime`, `date`, and `time`, and `Decimal`, `Enum` objects are seamlessly stored as TEXT (ISO-8601) giving them support for SQLite's native date functions. Buffer protocols (`bytes`, `bytearray`, `memoryview`) are automatically adapted to BLOBs.

Additionally, data structures and objects that can be serialized by `msgspec` are natively supported as fallback JSON columns. This includes `list`, `dict`, `set`, `tuple`, `dataclass`, `UUID`, etc.

In [ ]:
from enum import Enum
import datetime as dt
from decimal import Decimal
from dataclasses import dataclass

class ColorEnum(str, Enum):
    RED = "red"
    BLUE = "blue"

class StyleEnum(int, Enum):
    BOLD = 1
    ITALIC = 2
    UNDERLINE = 3

from typing import TypedDict

class SourceVerTD(TypedDict):
    source: str
    version: int

@dataclass
class LocationDC:
    lat: float
    lng: float


class AdvancedTypesModel(TableRow):
    # Native
    score: float

    # Datetimes
    created_at: dt.datetime
    precision_value: Decimal

    # Buffers
    raw_data: bytes

    # JSON fallbacks
    tags: set[str]
    metadata: dict
    source_ver: SourceVerTD
    new_style: StyleEnum
    favorite_color: ColorEnum
    colors: list[ColorEnum]
    location: LocationDC

engine.ensure_table_created(AdvancedTypesModel)

row = engine.insert(AdvancedTypesModel(
    score=42.5,
    created_at=dt.datetime(2024, 6, 15, 12, 0, 0, tzinfo=dt.UTC),
    precision_value=Decimal("123.456"),
    raw_data=b"hello_world",
    tags={"urgent", "new"},
    metadata={"kind": "demo", "featured": True},
    source_ver={"source": "api", "version": 1},
    new_style=StyleEnum.BOLD,
    favorite_color=ColorEnum.RED,
    colors=[ColorEnum.RED, ColorEnum.BLUE],
    location=LocationDC(lat=40.7128, lng=-74.0060),
))

fetched = engine.find(AdvancedTypesModel, row.id)
fetched

## SQLite3 supports JSON extensions

In [ ]:
from typing import TypedDict

class Stats(TypedDict):
    spell: str
    level: int

class Character(TableRow):
    name: str
    stats: Stats # JSON field

engine.ensure_table_created(Character)

engine.insert(Character('Harbel', {'spell': 'Fireball', 'level': 3}))
engine.insert(Character('Quenswen', {'spell': 'Waterspout', 'level': 27}))
engine.insert(Character('Ruthbag', {'spell': 'Fireball', 'level': 12}))

for c in engine.select(Character, t"{Character.stats} ->> '$.spell' = 'Fireball'"):
    print(f"{c.name} has a fireball at level {c.stats['level']}")

# Performance scenarios
Every call to insert real full trip to the db. The data is ready to be queried immediately, in SQLAlchemy parlance, 'flushed'. Committig ends the implicit transaction and ensures that the data is persisted to disk. Data is then avialable to other connections e.g. other worker processes

Because the db and app share a process, the performance is good enough that you can basically ignore the N+1 problem. This also simplifies implementation of this library, no need to track session etc. It also simplifies your app as data is syncronized immediately with the database, thus eliminates the need for a stateful cache, a source off many bugs and complexity.

In [ ]:
engine.connection.exec_trace = None # disable echo SQL

## Insert many (17,000 rows)

In [ ]:
rows = [MyModel("foo", dt.datetime.now(), random()*100) for _ in range(17000)]

In [ ]:
with engine.connection:  # transaction
    for r in rows:
        engine.insert(r)

## Update many (17,000 rows)

In [ ]:
updates = [{'id': row.id, 'score': random()*100, 'date': dt.datetime.now()} for row in rows]

In [ ]:
with engine.connection:  # transaction
    for u in updates:
        engine.update(MyModel, u['id'], date=u['date'], score=u['score'])


## Query many

In [ ]:
def print_30_per_line(ss: list[str]):
    for i,s in enumerate(ss, 1):
        print(s, end=" ")
        if i % 30 == 0:
            print()
    print()

rows = engine.select(MyModel, MyModel.score > 95.7)
print_30_per_line([f"{r.score:5.1f}" for r in rows])

## Giant Recursive BOM

In [ ]:
class BOM(TableRow):
    name: str
    value: float
    child_a: BOM | None
    child_b: BOM | None

engine.ensure_table_created(BOM)

from random import random, choice
node_count = 0
def generate_node_name_node(depth: int) -> str:
    alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    return f"{choice(alphabet)}{choice(alphabet)}{choice(alphabet)}{depth:05d}_{node_count}"


# create a giant BOM, of 15 levels deep
def create_bom(depth: int) -> BOM:
    global node_count
    node_count += 1

    if depth == 1:
        child_a = None
        child_b = None
    else:
        child_a = create_bom(depth-1)
        child_b = create_bom(depth-1)

    return BOM(generate_node_name_node(depth), random()*1000 - 500, child_a, child_b)

root = create_bom(13)
print(f"Created a BOM with {node_count} nodes")

In [ ]:
from dataclasses import replace

def save_bom_recursive(node: BOM) -> BOM:
    """Save BOM tree leaf-first (children must be saved before parents)."""
    return engine.insert(replace(
        node,
        child_a=save_bom_recursive(node.child_a) if node.child_a else None,
        child_b=save_bom_recursive(node.child_b) if node.child_b else None
    ))

with engine.connection:  # transaction
    inserted_root = save_bom_recursive(root)

print(f"Inserted BOM with id: {inserted_root.id}")

In [ ]:
recovered_root = engine.find(BOM, inserted_root.id)
assert recovered_root is not None

def count_nodes(node: BOM | None) -> int:
    if node is None:
        return 0
    return 1 + count_nodes(node.child_a) + count_nodes(node.child_b)

# counting the node lazily traverses the whole tree, one query at a time
print(f"Recovered BOM with {count_nodes(recovered_root)} nodes")

In [ ]:
import math

import matplotlib.pyplot as plt
import networkx as nx

def add_nodes_edges(G: nx.Graph, node: BOM | None):
    if node is None:
        return

    G.add_node(node.id, label=node.name)
    if node.child_a is not None:
        G.add_edge(node.id, node.child_a.id)
        add_nodes_edges(G, node.child_a)

    if node.child_b is not None:
        G.add_edge(node.id, node.child_b.id)
        add_nodes_edges(G, node.child_b)

def hierarchical_tree_layout(G, root_node):
    pos = {}

    # Build adjacency list from the graph
    adj = {node: list(G.neighbors(node)) for node in G.nodes()}

    # BFS to determine levels and children
    from collections import deque
    queue = deque([(root_node, 0)])
    visited = {root_node}
    levels = {}
    children = {node: [] for node in G.nodes()}

    while queue:
        node, level = queue.popleft()
        levels[node] = level

        for neighbor in adj[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                children[node].append(neighbor)
                queue.append((neighbor, level + 1))

    # Position nodes level by level
    def position_subtree(node, level, angle_start, angle_end):
        # Position current node
        if level == 0:
            pos[node] = (0, 0)  # Root at center
        else:
            angle = (angle_start + angle_end)
            radius = level * 0.8  # Increase radius per level
            x = radius * math.cos(angle)
            y = radius * math.sin(angle)
            pos[node] = (x, y)

        # Position children
        kids = children[node]
        if kids:
            angle_span = min(angle_end - angle_start, 2 * math.pi / max(1, len(kids)))
            angle_per_child = angle_span / len(kids)

            for i, child in enumerate(kids):
                child_angle_start = angle_start + i * angle_per_child
                child_angle_end = child_angle_start + angle_per_child
                position_subtree(child, level + 1, child_angle_start, child_angle_end)

    # Start positioning from root
    position_subtree(root_node, 0, 0, 2 * math.pi)
    return pos


G = nx.Graph()
add_nodes_edges(G, recovered_root)

plt.figure(figsize=(10, 10))
nx.draw(G, hierarchical_tree_layout(G, recovered_root.id),
    node_size=6, width=0.2, node_color="blue",
    with_labels=node_count<1200,
    labels=nx.get_node_attributes(G, "label"),
    )
plt.show()

# Model Class
The model class itself hold info about the table/model/types/etc

See `tuplesaver/model.py` and `RowMeta` for more details.

In [ ]:
Team.__fields__ # just one of a handfule of attrs defined on the model class, populated during compilation.
